# Projet Data Science — Diabetes 130-US Hospitals (1999–2008)
## Étape 0 — Business Understanding

Première phase de la méthode CRISP-DM : comprendre le problème métier avant de toucher aux données.

---
### 0.1 Les données

- **Source** : base clinique *Health Facts* (Cerner), **130 hôpitaux américains**, **1999–2008**, publiée par Strack et al. (2014) sur l'UCI Repository.
- **Population** : hospitalisations de **patients diabétiques**, de **1 à 14 jours**, avec analyses de laboratoire et médicaments administrés.
- **Une ligne = un séjour** (et non un patient : un même patient peut revenir plusieurs fois).
- **Variables** : démographie (âge, sexe, origine), parcours (type d'admission, destination de sortie, assurance), intensité du séjour (durée, analyses, médicaments), historique de l'année précédente (urgences, hospitalisations), diagnostics (codes ICD-9), examens glycémiques (HbA1c), 23 médicaments du diabète, et la cible `readmitted`.

In [ ]:
import pandas as pd

df = pd.read_csv('diabetic_data.csv', na_values='?', keep_default_na=False, low_memory=False)

print(f"Séjours                       : {len(df):,}".replace(',', ' '))
print(f"Patients distincts            : {df['patient_nbr'].nunique():,}".replace(',', ' '))
print(f"Réadmis en moins de 30 jours  : {(df['readmitted'] == '<30').mean():.1%}")
print(f"HbA1c non mesurée             : {(df['A1Cresult'] == 'None').mean():.1%}")

: 

---
### 0.2 Contexte

- Les patients sont **diabétiques mais souvent hospitalisés pour autre chose** (le diagnostic principal le plus fréquent est l'insuffisance cardiaque) : le diabète est géré « au passage ».
- L'**HbA1c** est l'examen de référence du suivi du diabète (glycémie moyenne des 3 derniers mois). Il n'est pas réalisé dans plus de 8 séjours sur 10.
- Une **réadmission en moins de 30 jours** est un indicateur de qualité des soins (sortie trop précoce, traitement mal ajusté, suivi insuffisant). Elle est coûteuse, risquée pour le patient, et pénalisée aux États-Unis depuis 2012 (*Hospital Readmissions Reduction Program*).

---
### 0.3 Problématique et objectifs

**Problème métier** : l'hôpital veut réduire les réadmissions précoces, mais ne peut pas offrir un suivi renforcé (appel après la sortie, rendez-vous rapproché, éducation thérapeutique) à tous les patients. **Qui faut-il cibler en priorité ?**

| | Segmentation | Classification |
|---|---|---|
| **Question** | Quels profils de patients existe-t-il, et sont-ils plus ou moins réadmis ? | Peut-on prédire, **le jour de la sortie**, une réadmission en moins de 30 jours ? |
| **Méthode** | Non supervisée (K-Means) | Supervisée (2 à 3 modèles) |
| **Utilité** | Adapter les parcours de soins à chaque profil | Signaler les patients à suivre en priorité |

**Cible** : `target = 1` si `readmitted = '<30'`, sinon 0. Le seuil de 30 jours est celui de l'indicateur de qualité ; les réadmissions plus tardives dépendent surtout de l'évolution de la maladie.

La prédiction se fait à la sortie : seules les informations connues à ce moment peuvent servir au modèle (`readmitted` et les identifiants sont exclus).

---
### 0.4 Parties prenantes

| Partie prenante | Attente |
|---|---|
| Direction / service qualité | Réduire le taux de réadmission et les pénalités |
| Médecins | Un outil d'aide à la décision compréhensible |
| Coordinateurs de sortie | Une liste courte de patients prioritaires |
| Payeurs (Medicare, assurances) | Moins de réhospitalisations évitables |
| Patients | Moins de retours à l'hôpital, données protégées |

---
### 0.5 Critères de succès

La cible est déséquilibrée (≈ 9 % de réadmis après nettoyage) : un modèle qui prédit toujours « non réadmis » atteint ≈ 91 % d'accuracy sans détecter personne. L'accuracy est donc écartée.

| Critère | Objectif fixé par l'équipe |
|---|---|
| **Recall** (part des réadmis détectés) | ≥ 0,60 — rater un patient à risque coûte plus cher qu'un suivi inutile |
| **ROC-AUC** (0,5 = hasard) | ≥ 0,65 |
| **F1-score** | Nettement supérieur au modèle naïf |
| **Clusters** | Choisis par coude + silhouette, interprétables, avec des taux de réadmission différents |
| **Préparation** | Aucune fuite de données, aucune valeur manquante, chaque choix justifié |

L'exploration montre qu'aucune variable n'est fortement liée à la cible (corrélation max 0,14) : des performances **modestes** sont attendues.

---
### 0.6 Contraintes et risques

| Contrainte / risque | Réponse |
|---|---|
| Patients présents plusieurs fois → fuite de données | Un seul séjour par patient |
| Patients décédés ou en soins palliatifs (non réadmissibles) | Séjours supprimés |
| Cible déséquilibrée | Recall, F1, ROC-AUC ; split stratifié ; pondération des classes |
| Pas de dates dans les données | Ordre des `encounter_id` supposé chronologique |
| Données anciennes, américaines, population sélectionnée | Résultats non généralisables tels quels (limite assumée) |
| Variables sensibles (`race`, `gender`, `payer_code`) | Le modèle sert à proposer plus de suivi, jamais à en refuser |
| Corrélation ≠ causalité | Résultats présentés comme des associations |
| Délai (4 octobre), présentation de 10 minutes, travail à 5 | Un fichier transmis par étape, conventions de lecture communes |